# Proyecto de Primer Bimestre  
## Sistema de Recuperación de Información  

**Integrantes:** Bautista Alexis - Correa Francisco  
**Fecha de entrega:** 1 de junio de 2026

### a. Construcción del índice

In [1]:
import preprocesamiento

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Leer un corpus de documentos en texto plano.

In [2]:
corpus = preprocesamiento.cargar_corpus()

Se encontraron 8 archivos CSV en el directorio.
Carga completa. El corpus tiene un total de 51081 documentos individuales.


Procesamiento básico: tokenización, normalización y remoción de stopwords.

In [3]:
corpus_procesado = []
for doc in corpus:
    tokens_doc = preprocesamiento.preprocesar(doc)
    corpus_procesado.append(tokens_doc)

print(f"Se han preprocesado {len(corpus_procesado)} documentos.")

Se han preprocesado 51081 documentos.


In [4]:
#Verificacion de que se proceso
# print(corpus_procesado)

Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia.

In [5]:
import indice_invertido

In [6]:
indice = indice_invertido.crear_indice_invertido(corpus_procesado)

El indice a sido creado correctamente


In [7]:
# prueba rapida
if "bank" in indice:
    for doc_id, freq in list(indice["bank"].items())[:5]:
        print(f"{doc_id}: {freq}")
else:
    print("El término no existe en el corpus")

4: 1
9: 15
12: 5
13: 2
14: 3


**Ejemplos de interpretacion:**

4: 1 -> En el documento con el ID 4, la palabra "bank" aparece 1 vez.  
9: 15 -> En el documento con el ID 9, la palabra "bank" se repite 15 veces.  
12: 5 -> En el documento con el ID 12, la palabra "bank" se encuentra 5 veces.

In [8]:
# prueba rapida
print(indice.get("DDada", "El término no existe en el corpus"))

El término no existe en el corpus


### b. Modelo de recuperación

Implementar recuperación basada en similitud Jaccard utilizando vectores binarios

In [9]:
import modelos

In [10]:
query = input ("Ingrese una consulta: ")

In [11]:
ranking = modelos.recuperar_jaccard(query, corpus_procesado)

print(f"Resultados para: '{query}'")
for doc_id, score in ranking[:5]: # Mostrar el top 5
    
    print(f"Documento ID: {doc_id} | Similitud: {score * 100:.2f}%")

Resultados para: 'dog'
Documento ID: 22060 | Similitud: 5.00%
Documento ID: 46599 | Similitud: 5.00%
Documento ID: 22064 | Similitud: 4.00%
Documento ID: 46603 | Similitud: 4.00%
Documento ID: 30416 | Similitud: 3.23%


Implementar recuperación basada en similitud de coseno utilizando TF-IDF

In [12]:
ranking_tfidf = modelos.recuperar_tfidf(query, corpus_procesado)

print(f"Resultados TF-IDF para: '{query}'")
for doc_id, score in ranking_tfidf[:5]: # Mostrar el top 5
    # similitud coseno entre 0 y 1
    print(f"Documento ID: {doc_id} | Similitud Coseno: {score:.4f}")

Resultados TF-IDF para: 'dog'
Documento ID: 7479 | Similitud Coseno: 0.3123
Documento ID: 19641 | Similitud Coseno: 0.3123
Documento ID: 44183 | Similitud Coseno: 0.3123
Documento ID: 22060 | Similitud Coseno: 0.2475
Documento ID: 46599 | Similitud Coseno: 0.2475


Implementar recuperación con BM25.

In [13]:
ranking_bm25 = modelos.recuperar_bm25(query, corpus_procesado, indice)

print(f"Resultados BM25 para: '{query}'")
for doc_id, score in ranking_bm25[:5]:
    print(f"Documento ID: {doc_id} | Score BM25: {score:.4f}")

Resultados BM25 para: 'dog'
Documento ID: 7479 | Score BM25: 12.5991
Documento ID: 19641 | Score BM25: 12.5991
Documento ID: 44183 | Score BM25: 12.5991
Documento ID: 22060 | Score BM25: 10.9901
Documento ID: 46599 | Score BM25: 10.9901


**Interpretación de Métricas BM25**

* **Scores no normalizados:** Los valores obtenidos (ej. 7.1117) no son porcentajes ni probabilidades (0-100%). Son métricas relativas que solo sirven para comparar qué documento es más relevante que otro dentro de la *misma* consulta.
* **Empates matemáticos:** Los scores idénticos ocurren cuando los documentos son textos duplicados, o cuando coinciden exactamente en su longitud total y en la cantidad de veces que repiten los términos buscados.
* **Criterios de relevancia:** Un score alto indica que el documento contiene las palabras más "raras" de la búsqueda (alto IDF), las menciona de forma natural sin hacer spam (saturación de TF) y es un texto relativamente conciso (penalización a documentos muy largos).

### c. Interfaz básica

### d. Recuperación semántica con embeddings

• Generar embeddings para los documentos del corpus utilizando un modelo preentrenado.  
• Generar embeddings para las consultas de texto libre.  
• Almacenar los embeddings en una base de datos vectorial, como ChromaDB o FAISS.  
• Recuperar los documentos más similares usando búsqueda vectorial.  
• Mostrar un ranking de resultados basado en similitud vectorial.  

Para modelos clasicos como TF-IDF, BM25 es necesrio tener los tokens limpios y el stemming (ej. ["japan", "bank", "tax"]). Sin embargo, a los modelos semánticos (Transformers) les hace daño el preprocesamiento agresivo. Estos modelos necesitan leer el texto con su sintaxis, puntuación y conectores (stop words) para entender el contexto real de la oración.

Por esto se usara el corpus original para esta seccion. Ademas se decidio usar la base de datos vectorial FAISS. Por ultimo se decidio usar el modelo all-MiniLM-L6-v2 ya que es el estándar de la industria para este tipo de proyectos académicos porque es extremadamente rápido, pesa poco y ofrece una precisión altísima para representar oraciones en inglés.

In [14]:
import modelo_semantico

modelo_transformer, base_vectorial_faiss = modelo_semantico.construir_indice_faiss(corpus)

Cargando el modelo preentrenado 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings para 51081 documentos...
(Esto puede tomar unos minutos dependiendo del procesador)


Batches:   0%|          | 0/1597 [00:00<?, ?it/s]

Construyendo el índice FAISS (Dimensión: 384)...
Índice FAISS completado con 51081 vectores.


In [19]:
#prueba
mi_busqueda = "cat in blue house house"

ranking_semantico = modelo_semantico.recuperar_semantico(
    query_texto=mi_busqueda, 
    modelo=modelo_transformer, 
    indice_faiss=base_vectorial_faiss, 
    top_k=5
)

print(f"Resultados Semánticos para: '{mi_busqueda}'")
for doc_id, score in ranking_semantico:
    print(f"Documento ID: {doc_id} | Similitud Semántica: {score:.4f}")

Resultados Semánticos para: 'cat in blue house house'
Documento ID: 26401 | Similitud Semántica: 0.3986
Documento ID: 50174 | Similitud Semántica: 0.3986
Documento ID: 28954 | Similitud Semántica: 0.3932
Documento ID: 34440 | Similitud Semántica: 0.3932
Documento ID: 5376 | Similitud Semántica: 0.3931


## Evaluación de modelos

En esta sección se usa únicamente el conjunto de prueba para construir una comparación entre Jaccard, TF-IDF, BM25 y recuperación semántica.

Como este corpus no trae consultas separadas ni un archivo de qrels externo, se usa la columna `topics` como criterio de relevancia: un documento es relevante para una consulta si contiene ese tema.

In [21]:
import importlib
import evaluacion
import modelo_semantico

importlib.reload(modelo_semantico)
importlib.reload(evaluacion)

comparacion_modelos, resultados_por_modelo, qrels_eval, resumen_por_modelo = evaluacion.ejecutar_evaluacion_completa(
    modelo_semantico_existente=modelo_transformer,
    corpus_path="corpus",
    archivo_test="ModApte_test.csv",
    consultas_eval=["earn", "acq", "crude", "trade", "money-fx", "grain", "cocoa"],
    k=10,
    top_k_semantico=20,
)

comparacion_modelos

El indice a sido creado correctamente
Reutilizando el modelo preentrenado ya cargado...
Generando embeddings para 3023 documentos...
(Esto puede tomar unos minutos dependiendo del procesador)


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

Construyendo el índice FAISS (Dimensión: 384)...
Índice FAISS completado con 3023 vectores.


,modelo,map@10,precision_promedio@10,recall_promedio@10
0,tfidf,0.001677,0.114286,0.005650
1,bm25,0.001180,0.100000,0.003745
2,semantico,0.000325,0.057143,0.000674
3,jaccard,0.000053,0.028571,0.000264


## f. Comparación de modelos

En la comparación global, **TF-IDF** obtuvo el mejor desempeño, seguido por **BM25**. Esto indica que, para este corpus y estas consultas de prueba, los modelos basados en coincidencia léxica capturan mejor la relevancia que la recuperación semántica.

- **Jaccard** fue el modelo más débil, como era esperado, porque solo considera presencia o ausencia de términos y no pondera frecuencia ni rareza.
- **TF-IDF** funcionó mejor en consultas con términos específicos y claramente presentes en los documentos relevantes, por ejemplo `earn`, `acq`, `trade` y `crude`.
- **BM25** también respondió bien en consultas basadas en palabras clave, pero quedó ligeramente por debajo de TF-IDF en este experimento.
- **Recuperación semántica con embeddings** fue la que obtuvo los valores más bajos aquí. Esto sugiere que, en este conjunto y con estas consultas, el modelo semántico no superó a los modelos clásicos.

La recuperación semántica suele ser más útil cuando la consulta es más natural, parafraseada o usa sinónimos que no aparecen exactamente en los documentos. En cambio, cuando la consulta coincide con términos temáticos muy concretos del corpus, como en este caso, los modelos léxicos suelen rendir mejor.

En resumen, **BM25 y TF-IDF son las mejores opciones para este corpus de prueba**, mientras que la recuperación semántica podría mejorar en consultas más abiertas o menos literales.